In [ ]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from datetime import datetime
import copy
import collections
from astropy.cosmology import Planck13  # Planck (2014) Paper XVI, Table 5

import gc
gc.enable()

import sys
sys.path.append('..') 

from utils import Vcom, get_comoving_distance_h, c, z_min
# from data_stat_func import wprp, dPdN
from data_stat_func import weighted_wprp
from data_stat_func import wprp

from halotools.mock_observables import apply_zspace_distortion
from halotools.mock_observables import return_xyz_formatted_array
import halotools.mock_observables as htmo

import Corrfunc
from Corrfunc.mocks.DDrppi_mocks import DDrppi_mocks
from Corrfunc.utils import convert_rp_pi_counts_to_wp
import matplotlib.pyplot as plt

from scipy.stats import gaussian_kde

In [ ]:
z_max_arr  = np.array((       47650, 31900, 19250, 12500))/ c
M_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0))
C_thr_arr  = np.array([[0.0,  0.82,  0.94,  0.97,   1.2,],
                       [0.0,  0.72,  0.90,  0.94,   1.2,],
                       [0.0,  0.55,  0.85,  0.92,   1.2,],
                       [0.0,  0.50,  0.73,  0.87,   1.2,],])

M_thr_list  = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
C_thr_list  = np.array([['0_p82'  , 'p82_p94', 'p94_p97', 'p97_1p2',],
                        ['0_p72'  , 'p72_p90', 'p90_p94', 'p94_1p2',],
                        ['0_p55'  , 'p55_p85', 'p85_p92', 'p92_1p2',],
                        ['0_p50'  , 'p50_p73', 'p73_p87', 'p87_1p2',],])

Mr_thr_text = [r'$M_r: [-22,-21]$',r'$M_r: [-21,-20]$',r'$M_r: [-20,-19]$',r'$M_r: [-19,-18]$',]
Co_thr_text = np.array([[r'g-r: [0,0.82]', r'g-r: [0.82,0.94]', r'g-r: [0.94,0.97]', r'g-r: [0.97,1.2]',],
                        [r'g-r: [0,0.72]', r'g-r: [0.72,0.90]', r'g-r: [0.90,0.94]', r'g-r: [0.94,1.2]',],
                        [r'g-r: [0,0.55]', r'g-r: [0.55,0.85]', r'g-r: [0.85,0.92]', r'g-r: [0.92,1.2]',],
                        [r'g-r: [0,0.50]', r'g-r: [0.50,0.73]', r'g-r: [0.73,0.87]', r'g-r: [0.87,1.2]',],])

In [ ]:
from matplotlib import cm

colors4 = [cm.YlGnBu(55)  , cm.YlGnBu(80)  , cm.YlGnBu(120) , cm.YlGnBu(150) , cm.YlGnBu(180)]


halomass_bins     = np.linspace(9,15,61)
hod_halomassbin_x = (halomass_bins[:-1] + halomass_bins[1:])/2

#### Mr_C_bins(~2h each)

In [ ]:
# gal_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")
# gal_data[1].data

In [ ]:
# gal_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")
# gal_data[1].header

In [ ]:
from astropy.io import fits

### halotable
halo_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_HaloInfo.fits")[1].data
# hdul_halo[1].header
Box_MH_halotab  = halo_data['logMh'] # host halo mass, log(Msun/h)
Box_Host_HaloID = halo_data['haloID']

### galaxytable
gal_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data

output_dict1 = collections.defaultdict(list)
output_dict2 = collections.defaultdict(list)

n_g          = np.zeros((4,4))
cen_Mh_med   = np.zeros((4,4))
sat_Mh_med   = np.zeros((4,4))
fsat         = np.zeros((4,4))
Nsat_perhost = np.zeros((4,4))

#########################################################################################################################
for j in range(4):
    for i in range(4):
        mask = np.logical_and(  np.logical_and(M_thr_arr[j]   < gal_data['Mr'] , gal_data['Mr']  < M_thr_arr[j+1]  ),
                                np.logical_and(C_thr_arr[j,i] < gal_data['gmr'], gal_data['gmr'] < C_thr_arr[j,i+1]) )

        Box_MH_gal    = gal_data['logMh'][mask] # host halo mass, log(Msun/h)
        Box_Gal_Type  = gal_data['type'][mask]
        Box_Gal_Type  = gal_data['type'][mask]
        Box_HaloID    = gal_data['haloID'][mask]
        
        Cen_mask      = (Box_Gal_Type == 0) | (Box_Gal_Type == 1) # 0/1 - pseudo-blue/red central galaxy
        Sat_mask      = (Box_Gal_Type == 2) | (Box_Gal_Type == 3) # 2/3 - pseudo-blue/red satellite galaxy

        n_g[j,i]          = np.count_nonzero(mask)/1000**3 # (Mpc/h)^-3
        cen_Mh_med[j,i]   = np.median(Box_MH_gal[Cen_mask])
        sat_Mh_med[j,i]   = np.median(Box_MH_gal[Sat_mask])
        fsat[j,i]         = np.count_nonzero(Sat_mask)/np.count_nonzero(mask)
        Nsat_perhost[j,i] = np.count_nonzero(Sat_mask)/np.count_nonzero(np.unique(Box_HaloID[Sat_mask]))
        #################################################################################################################

    #     Ncen     = np.zeros(len(hod_halomassbin_x))
    #     Nsat     = np.zeros(len(hod_halomassbin_x))
    #     Nhalo    = np.zeros(len(hod_halomassbin_x))

    #     for k in range(len(halomass_bins)-1):
    #         lower_mh    = halomass_bins[k]
    #         upper_mh    = halomass_bins[k+1]
    #     ### galaxytable
    #         AB_Mh_mask  = (lower_mh < Box_MH_gal) & (Box_MH_gal < upper_mh)

    #         Ncen[k]     = np.count_nonzero(Cen_mask & AB_Mh_mask)
    #         Nsat[k]     = np.count_nonzero(Sat_mask & AB_Mh_mask)
    #     ### halotable
    #         Box_Mh_mask = (lower_mh < Box_MH_halotab) & (Box_MH_halotab < upper_mh)
    #         Nhalo[k]    = np.count_nonzero(np.unique(Box_Host_HaloID[Box_Mh_mask]))

    #     HOD_cen  = Ncen/Nhalo
    #     HOD_sat  = Nsat/Nhalo
                
    #     output_dict1['HOD_cen_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = HOD_cen
    #     output_dict2['HOD_sat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = HOD_sat

    # np.savez('../CCMD_mag_bin/HOD/CCMD_Box_Mr_C_HOD_cen.npz', **output_dict1)
    # np.savez('../CCMD_mag_bin/HOD/CCMD_Box_Mr_C_HOD_sat.npz', **output_dict2)

In [ ]:
# np.save( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_n_g'         , n_g          )
# np.save( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_cen_Mh_med'  , cen_Mh_med   )
# np.save( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_sat_Mh_med'  , sat_Mh_med   )
# np.save( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_fsat'        , fsat         )
# np.save( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_Nsat_perhost', Nsat_perhost )

### Plot

In [ ]:
CCMD_box_HOD_cen = np.load('../CCMD_mag_bin/HOD/CCMD_Box_Mr_C_HOD_cen.npz')
CCMD_box_HOD_sat = np.load('../CCMD_mag_bin/HOD/CCMD_Box_Mr_C_HOD_sat.npz')

In [ ]:
n_g          = np.load( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_n_g.npy'         )
cen_Mh_med   = np.load( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_cen_Mh_med.npy'  )
sat_Mh_med   = np.load( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_sat_Mh_med.npy'  )
fsat         = np.load( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_fsat.npy'        )
Nsat_perhost = np.load( '../CCMD_mag_bin/HOD/CCMD_box_Mr_C_Nsat_perhost.npy')

In [ ]:
fig, ax = plt.subplots(1, 4, figsize = (20, 4), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)


for j in range(4):
    cen_stat = np.zeros(len(hod_halomassbin_x))
    sat_stat = np.zeros(len(hod_halomassbin_x))
    for i in range(4):
    ### CCMD Box #############################################################################################################
        cen_stat = cen_stat + CCMD_box_HOD_cen['HOD_cen_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        sat_stat = sat_stat + CCMD_box_HOD_sat['HOD_sat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

    ax[j].plot(hod_halomassbin_x, cen_stat, color = 'forestgreen', alpha = 1, lw = 3, ls = '-' , label = 'CCMD Box cen')
    ax[j].plot(hod_halomassbin_x, sat_stat, color = 'forestgreen', alpha = 1, lw = 3, ls = '--', label = 'CCMD Box sat')
#############################################################################################################   
    ax[j].loglog()
    ax[j].tick_params(axis = 'x', direction = 'in', which = u'both')
    ax[j].tick_params(axis = 'y', direction = 'in', which = u'both')
    ax[j].set_xlim(10.5,14.5)
    ax[j].set_ylim(0.015, 20)
    
for j in range(4):
    ax[0].set_ylabel(r'$\log\langle N(M_{vir}) \rangle$',fontsize = 16,)
    ax[j].set_yscale('log')
    ax[j].set_title(Mr_thr_text[j],  fontsize=16,)
    ax[j].set_xlabel(r'$\log (M_{\rm vir}/h^{-1}M_\odot )$',fontsize = 16,)
    ax[j].set_xscale('linear')

legend = ax[0].legend(fontsize = 16, loc = 'upper left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('forestgreen')

# plt.suptitle(r'$N_g^8$ bottom $30\%$', y =0.95)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
CCMD_box_xi_r       = np.load('./CCMD_box_Mr_C_xi.npz')
CCMD_box_xi_r_bot20 = np.load('./CCMD_box_Mr_C_xi_bottom_20.npz')
CCMD_box_xi_r_top80 = np.load('./CCMD_box_Mr_C_xi_top_80.npz')

xi_mm_r       = np.array([0.3257651 , 0.08987048, 0.03611578, 0.01660525, 0.00844831])

In [ ]:
def Mh_vlines(Mh_bin_x, HOD_stat, Mh_med):
    x = Mh_bin_x
    y = HOD_stat

    # 选择竖线的横坐标位置
    x_line = Mh_med

    # 查找与竖线相交的点
    index = np.searchsorted(x, x_line)

    # 确保 index 在有效范围内
    if index > 0 and index < len(x):
        # 获取相邻的两个点
        x1, y1 = x[index - 1], y[index - 1]
        x2, y2 = x[index], y[index]
        
        # 检查是否有交点
        if (y1 - y2) * (x_line - x1) < 0:  # 判断交点是否在y轴范围内
            # 计算交点的纵坐标（线性插值）
            y_line = y1 + (y2 - y1) * (x_line - x1) / (x2 - x1)
        elif (y1 - y2) * (x_line - x1) > 0:  # 判断交点是否在y轴范围内
            # 计算交点的纵坐标（线性插值）
            y_line = y1 + (y2 - y1) * (x_line - x1) / (x2 - x1)
        else:
            y_line = None  # 没有交点
    else:
        y_line = None  # 超出范围，无法找到交点

    return y_line

In [ ]:
fig, ax = plt.subplots(4, 4, figsize = (20, 12), sharex = True, sharey = True, dpi=100)
plt.subplots_adjust(hspace = 0, wspace = 0)

x_add = [0.1, -0.2, -0.2, -0.1]

for j in range(4):
    for i in range(4):
    ### CCMD Box #############################################################################################################
        cen_stat = CCMD_box_HOD_cen['HOD_cen_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        sat_stat = CCMD_box_HOD_sat['HOD_sat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

        ax[j,i].plot(hod_halomassbin_x, cen_stat, color = 'forestgreen', alpha = 1, lw = 2, ls = '-' , label = 'Central')
        ax[j,i].plot(hod_halomassbin_x, sat_stat, color = 'forestgreen', alpha = 1, lw = 3, ls = '--', label = 'Satellite')

        ### Mh median central ####################################################################################################
        HOD_temp= Mh_vlines(hod_halomassbin_x, cen_stat, cen_Mh_med[j,i]) # interpolation
        ax[j,i].vlines(cen_Mh_med[j,i], ymin = 0.0, ymax = HOD_temp, ls = '-' , color = 'gray', alpha=1)
        # ax[j,i].axvline(Mh_temp, ls = '-' , color = 'dimgray', alpha=1)
        ### Mh median satellite ##################################################################################################
        HOD_temp= Mh_vlines(hod_halomassbin_x, sat_stat, sat_Mh_med[j,i]) # interpolation
        ax[j,i].vlines(sat_Mh_med[j,i], ymin = 0.0, ymax = HOD_temp, ls = '--', color = 'gray', alpha=1)
        # ax[j,i].axvline(Mh_temp, ls = '--', color = 'dimgray', alpha=1)

        # ax[j,i].text(10.7, 8.0, (r'$M_{h,med}^{cen}=$' + '{:.2f}'.format(cen_Mh_med[j,i])), fontsize = 14, color = 'black')
        # ax[j,i].text(10.7, 3.6, (r'$M_{h,med}^{sat}=$' + '{:.2f}'.format(sat_Mh_med[j,i])), fontsize = 14, color = 'black')
        ##########################################################################################################################
        ax[j,i].text(11.9, 16 , (r'$n_{\rm gal~}=$' + '{:.2e}'.format(n_g[j,i])), fontsize = 14, color = 'black')
        ax[j,i].text(11.9, 5.5, (r'$\overline{N_{\rm sat}}= $'+ '{:.2f}'.format(Nsat_perhost[j,i])), fontsize = 14, color = 'black')
        ax[j,i].text(11.9, 2.5, (r'$f_{\rm sat~~}= $'+ '{:.2f}'.format(fsat[j,i])), fontsize = 14, color = 'black')

        bias_g_all = np.sqrt(CCMD_box_xi_r['xi_r_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] / xi_mm_r)
        bias_g_bot = np.sqrt(CCMD_box_xi_r_bot20['xi_r_bot20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] / xi_mm_r)
        bias_g_top = np.sqrt(CCMD_box_xi_r_top80['xi_r_top80_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] / xi_mm_r)

        ax[j,i].text(10.7, 15 , r'$b_{\rm all~}= $' + '{:.2f}'.format(np.nanmean(bias_g_all)), fontsize=14, color = 'black')
        ax[j,i].text(10.7, 6  , r'$b_{\rm top} = $' + '{:.2f}'.format(np.nanmean(bias_g_top)), fontsize=14, color = 'black')
        ax[j,i].text(10.7, 2.5, r'$b_{\rm bot} = $' + '{:.2f}'.format(np.nanmean(bias_g_bot)), fontsize=14, color = 'black')


        # ax[j,i].text(13+x_add[i], 9, Co_thr_text[j,i], fontsize = 14)
#############################################################################################################   
        ax[j,i].semilogy()
        ax[j,i].tick_params(axis = 'x', direction = 'in', which = u'both')
        ax[j,i].tick_params(axis = 'y', direction = 'in', which = u'both')
        ax[j,i].tick_params(axis='both', which='major', width=2, length=6, labelsize = 24)
        # ax[j,i].tick_params(axis='both', which='minor', width=1, length=0, labelsize = 24)
        ax[j,i].tick_params(axis='x', which='minor', width=1, length=4, labelsize = 24)
        ax[j,i].tick_params(axis='y', which='minor', width=1, length=0, labelsize = 24)
        ax[j,i].set_xlim(10.5,14.5)
        ax[j,i].set_ylim(0.005, 50)

        for spine in ax[j,i].spines.values():
            # 对于主图 - 分别调整每个边框
            ax[j,i].spines['top'].set_linewidth(2)
            ax[j,i].spines['bottom'].set_linewidth(2) 
            ax[j,i].spines['left'].set_linewidth(2)
            ax[j,i].spines['right'].set_linewidth(2)

        ax[j,i].set_xticks([11  , 12 , 13, 14], ['11', '12', '13', '14'])
        ax[j,i].set_yticks([0.01, 0.1, 1 , 10], ['-2', '-1', '0', '1'])

    
for j in range(4):
    # ax[j,0].set_ylabel(r'$\log\langle N(M_{vir}) \rangle$',fontsize = 16,)
    
    # ax[j,0].text(13.1, 9, Co_thr_text[j,0], fontsize = 14, color = 'royalblue')
    # ax[j,3].text(12.9, 9, Co_thr_text[j,3], fontsize = 14, color = 'red')

    # ax[j,0].text(13.4, 10 , 'bluest' , fontsize = 18, color = 'royalblue')
    # ax[j,3].text(13.2, 10 , 'reddest', fontsize = 18, color = 'red')
    # ax[j,3].text(14.7, 0.5, Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=18,)
    ax[j,0].text(10.7, 0.7, Mr_thr_text[j], fontsize=18,)
    # ax[j,3].text(10.7, 0.7, Mr_thr_text[j], fontsize=18,)

# for i in range(4):
#     ax[3,i].set_xlabel(r'$\log (M_{\rm vir}/h^{-1}M_\odot)$',fontsize = 16,)
    # ax[3,i].set_xscale('linear')

# ax[0,0].text(13.6, 6, 'bluest' , color = 'blue', fontsize = 16)
# ax[0,3].text(13.4, 6, 'reddest', color = 'red' , fontsize = 16)
legend = ax[0,0].legend(fontsize = 16, loc = 'lower left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('forestgreen')

ax[0,0].set_title(r'$blue$'   , fontsize = 24, color = 'royalblue')
ax[0,1].set_title(r'$green$'  , fontsize = 24, color = 'forestgreen')
ax[0,2].set_title(r'$red$'    , fontsize = 24, color = 'red')
ax[0,3].set_title(r'$redder$' , fontsize = 24, color = 'red')

fig.supxlabel(r'$\log (M_{\rm vir}/h^{-1}M_\odot)$', fontsize=28, fontweight='bold', y = 0.03)
fig.supylabel(r'$\log \langle N(M_{vir}) \rangle$' , fontsize=28, fontweight='bold', x = 0.075)

# plt.suptitle(r'$N_g^8$ bottom $30\%$', y =0.95)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
# fig, ax = plt.subplots(4, 4, figsize = (20, 12), sharex = True, sharey = True)
# plt.subplots_adjust(hspace = 0, wspace = 0)

# x_add = [0.1, -0.2, -0.2, -0.1]

# for j in range(4):
#     for i in range(4):
#     ### CCMD Box #############################################################################################################
#         cen_stat = CCMD_box_HOD_cen['HOD_cen_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#         sat_stat = CCMD_box_HOD_sat['HOD_sat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]

#         ax[3-i,3-j].plot(hod_halomassbin_x, cen_stat, color = 'forestgreen', alpha = 1, lw = 3, ls = '-' , label = 'Central')
#         ax[3-i,3-j].plot(hod_halomassbin_x, sat_stat, color = 'forestgreen', alpha = 1, lw = 3, ls = '--', label = 'Satellite')

#         ax[3-i,3-j].text(10.7, 8.0, (r'$n_{\rm gal}=$'     + '{:.3e}'.format(n_g[j,i]))       , fontsize = 14, color = 'black')
#         ax[3-i,3-j].text(10.7, 4.0, (r'$f_{sat~}=$'        + '{:.3f}'.format(fsat[j,i]))      , fontsize = 14, color = 'black')
#         ax[3-i,3-j].text(10.7, 1.8, (r'$M_{h,med}^{cen}=$' + '{:.2f}'.format(cen_Mh_med[j,i])), fontsize = 14, color = 'black')# + r'$h^{-1}M_\odot$'
#         ax[3-i,3-j].text(10.7, 0.8, (r'$M_{h,med}^{sat}=$' + '{:.2f}'.format(sat_Mh_med[j,i])), fontsize = 14, color = 'black')# + r'$h^{-1}M_\odot$'

#         ax[3-i,3-j].text(13+x_add[i], 9, Co_thr_text[j,i], fontsize = 14)
# #############################################################################################################   
#         ax[j,i].loglog()
#         ax[j,i].tick_params(axis = 'x', direction = 'in', which = u'both')
#         ax[j,i].tick_params(axis = 'y', direction = 'in', which = u'both')
#         ax[j,i].set_xlim(10.5,14.5)
#         ax[j,i].set_ylim(0.015, 20)
    
# for j in range(4):
#     ax[j, 0].set_ylabel(r'$\log\langle N(M_{vir}) \rangle$',fontsize = 16,)
#     ax[j, 0].set_yscale('log')
    
#     ax[3-0, 3-j].text(13.1, 9, Co_thr_text[j,0], fontsize = 14, color = 'royalblue')
#     ax[3-3, 3-j].text(12.9, 9, Co_thr_text[j,3], fontsize = 14, color = 'red')
#     ax[3-3, 3-j].text(11.7, 30, Mr_thr_text[j] , fontsize = 16,)#, rotation=90, va='center', ha='left'

# for i in range(4):
#     ax[3, i].set_xlabel(r'$\log (M_{\rm vir}/h^{-1}M_\odot)$',fontsize = 16,)
#     ax[3, i].set_xscale('linear')

# # ax[0,0].text(13.6, 6, 'bluest' , color = 'blue', fontsize = 16)
# # ax[0,3].text(13.4, 6, 'reddest', color = 'red' , fontsize = 16)
# legend = ax[0,3].legend(fontsize = 16, loc = 'lower left',frameon = False) 
# legend.get_texts()[0].set_color('forestgreen')
# legend.get_texts()[1].set_color('forestgreen')

# # plt.suptitle(r'$N_g^8$ bottom $30\%$', y =0.95)
# # plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
def resample_1percent(data):
    np.random.seed(42)
    
    n_total  = len(data)
    n_select = int(np.round(n_total * 0.02))
    # n_select = int(np.round(n_total * 0.0002))

    # 方法1：创建全False数组，然后随机设置True
    mask = np.zeros(n_total, dtype=bool)
    selected_indices = np.random.choice(n_total, size=n_select, replace=False)
    mask[selected_indices] = True

    result = data[mask]

    return result


In [ ]:

quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']
gal_data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + vars[0] + '_' + quads[0] + '_processed_catalog.fits')[1].data
len(gal_data['Mr'])

In [ ]:
sdss_bright1 = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data
np.shape(sdss_bright1)

In [ ]:
gal_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
Mr  = resample_1percent(gal_data['Mr']) # host halo mass, log(Msun/h)
len(Mr)

In [ ]:
gal_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
Mr  = resample_1percent(gal_data['Mr']) # host halo mass, log(Msun/h)
gmr = resample_1percent(gal_data['gmr'])

# ccmd_quad1 = fits.open('../CCMD_main_processed/CCMD_Main_Quad1_processed_catalog.fits')[1].data
# Mr  = ccmd_quad1['Mr'] 
# gmr = ccmd_quad1['gmr'] 

# sdss_bright1 = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data
# Mr  = sdss_bright1['Mr'] 
# gmr = sdss_bright1['gmr'] 
# mask = ~np.isnan(gmr)

# 计算密度
# kde   = gaussian_kde([Mr[mask], gmr[mask]])
kde   = gaussian_kde([gmr, Mr])
xgrid = np.linspace(0.0, 1.2, 100)  # 调整X范围
ygrid = np.linspace(-22, -18, 100)  # 调整Y范围
X, Y  = np.meshgrid(xgrid, ygrid)
Z     = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

# 创建图形
plt.figure(figsize=(10, 8))#,dpi=100

# # 绘制填充等高线 - 使用对数归一化
# cf = plt.contourf(X, Y, Z, 
#                  levels=np.logspace(-3, 0, 12),  # 12个对数间隔级别
#                 #  cmap='Spectral_r',
#                 #  alpha = 0,
#                  norm=LogNorm(vmin=1e-3, vmax=1))

# # 绘制等高线
# contours = plt.contour(X, Y, Z, 
#                       levels=np.logspace(-3, 0, 12),
#                       colors='k', linewidths=0.5, alpha=0.5)

contours = plt.contour(X, Y, Z, levels=9, colors='black')

# 添加标签
# plt.clabel(contours, inline=True, fontsize=8, fmt='%.3f')

# 添加分区线
# plt.axvline(x=-20, color='k', linestyle='--', linewidth=1)
# plt.axhline(y=0.5, color='k', linestyle=':', linewidth=1)
plt.plot([0.0,1.2],[-17.99,-17.99], c ='red', lw = 4, ls = '-')
plt.plot([0.0,1.2],[-19,-19], c ='red', lw = 4, ls = '-')
plt.plot([0.0,1.2],[-20,-20], c ='red', lw = 4, ls = '-')
plt.plot([0.0,1.2],[-21,-21], c ='red', lw = 4, ls = '-')
plt.plot([0.0,1.2],[-22,-22], c ='red', lw = 4, ls = '-')
plt.plot([0.0,0.0],[-22,-18], c ='red', lw = 4, ls = '-')
plt.plot([1.2,1.2],[-22,-18], c ='red', lw = 4, ls = '-')

plt.plot([0.50,0.50],[-18,-19], c ='red', lw = 3, ls = '-')
plt.plot([0.73,0.73],[-18,-19], c ='red', lw = 3, ls = '-')
plt.plot([0.87,0.87],[-18,-19], c ='red', lw = 3, ls = '-')

plt.plot([0.55,0.55],[-19,-20], c ='red', lw = 3, ls = '-')
plt.plot([0.85,0.85],[-19,-20], c ='red', lw = 3, ls = '-')
plt.plot([0.92,0.92],[-19,-20], c ='red', lw = 3, ls = '-')

plt.plot([0.72,0.72],[-20,-21], c ='red', lw = 3, ls = '-')
plt.plot([0.90,0.90],[-20,-21], c ='red', lw = 3, ls = '-')
plt.plot([0.94,0.94],[-20,-21], c ='red', lw = 3, ls = '-')

plt.plot([0.82,0.82],[-21,-22], c ='red', lw = 3, ls = '-')
plt.plot([0.94,0.94],[-21,-22], c ='red', lw = 3, ls = '-')
plt.plot([0.97,0.97],[-21,-22], c ='red', lw = 3, ls = '-')

# 设置坐标轴
plt.xlabel(r'$g-r$', fontsize=34, y = 0.01)
plt.ylabel(r'$M_r$', fontsize=34, labelpad = 10)
# plt.title('星系颜色-星等图 (对数密度)', fontsize=16)
plt.gca().invert_yaxis()

# 颜色条
# cbar = plt.colorbar(cf)
# cbar.set_label('对数密度', rotation=270, labelpad=20)
# plt.grid(alpha=0.3)

plt.yticks([-22, -21, -20, -19, -18.02], 
           ['-22', '-21', '-20', '-19', '-18'])
plt.xticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2], 
           ['0', '0.2', '0.4', '0.6', '0.8', '1.0', '1.2'])

plt.tick_params(
    axis='both',        # 同时设置x和y轴
    which='major',      # 设置主刻度
    direction='in',     # 刻度方向：'in'内侧, 'out'外侧, 'inout'两侧
    length=9,           # 刻度线长度
    width=3,            # 刻度线粗细
    labelsize=24        # 刻度标签字体大小
)

# plt.tick_params(axis='x', which='minor', width=2, length=5,) # X轴子刻度仍然显示
plt.tick_params(axis='y', which='major', length=0)  # Y轴子刻度隐藏，设置长度为0来隐藏

plt.xlim(-0.005, 1.205)
plt.ylim(-17.98, -22.02)

plt.tight_layout()
plt.show()

In [ ]:
def find_x(a, b):
    """
    x^a = b, return x
    """
    your_x = 10**(np.log10(b)/a)
    
    return your_x

In [ ]:
find_x(10, 5.7)

In [ ]:
find_x(10, 7.5)

In [ ]:
1.1**10

### CCMD bias

In [ ]:
def select_top80_in_random(x, weight, percentile=80):
    """
    根据权重筛选前 percentile% 的元素，
    若相同权重的元素超过比例限制，则随机抽取。
    
    参数:
        x: 数据数组 (np.array 或 list)
        weight: 权重数组 (np.array 或 list)
        percentile: 百分比（默认20）
    
    返回:
        筛选后的 x 的子集
    """
    x = np.array(x)
    weight = np.array(weight)
    
    # top80%, in percentile is 20%
    threshold = np.percentile(weight, 100-percentile)
    
    # 1. 直接取出权重严格大于阈值的元素
    mask_strictly_above = weight > threshold
    selected_above = x[mask_strictly_above]
    
    # 2. 处理权重等于阈值的元素
    mask_equal = weight == threshold
    x_equal = x[mask_equal]
    
    # 计算还需抽取的数量
    total_desired = int(len(x) * percentile / 100)  # 期望的总数量
    remaining_needed = total_desired - len(selected_above)
    
    # 随机抽取剩余需要的元素
    if remaining_needed > 0 and len(x_equal) > 0:
        np.random.seed(42)  # 固定随机种子（可选）
        selected_equal = np.random.choice(
            x_equal, 
            size=min(remaining_needed, len(x_equal)), 
            replace=False
        )
        result = np.concatenate([selected_above, selected_equal])
    else:
        result = selected_above
    
    return result

In [ ]:
data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
r_bin = np.array([5, 15, 25, 35, 45, 55])

output_dict1 = collections.defaultdict(list)

for j in range(4): # Mr[-22,-21] sample Ng8 is almost 0
    for i in range(4):
        mask = np.logical_and(  np.logical_and(M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1]))
        data_thr  = data[mask]

        gal_pos_thr = np.array([np.array(np.abs(data_thr['x'])%1000),
                                np.array(np.abs(data_thr['y'])%1000),    
                                np.array(np.abs(data_thr['z'])%1000)]).T
        weights = np.ones_like(data_thr['x'])
        
        CCMD_box_xi_r = Corrfunc.theory.xi(boxsize = 1000, nthreads = 50, binfile = r_bin, 
                                    X = gal_pos_thr[:, 0], Y = gal_pos_thr[:, 1], Z = gal_pos_thr[:, 2],
                                    weights=weights, weight_type='pair_product', output_ravg=True )['xi']
        
        output_dict1['xi_r_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = CCMD_box_xi_r
        np.savez('./CCMD_box_Mr_C_xi.npz', **output_dict1)


In [ ]:
from Ng8_func import N_g_8_theory
from Ng8_func import select_bottom20_in_random

In [ ]:
data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
r_bin = np.array([5, 15, 25, 35, 45, 55])

output_dict1 = collections.defaultdict(list)
output_dict2 = collections.defaultdict(list)

for j in range(4): # Mr[-22,-21] sample Ng8 is almost 0
    for i in range(4):
        mask = np.logical_and(  np.logical_and(M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1]))
        data_thr  = data[mask]

        gal_pos_thr = np.array([data_thr['x']%1000,
                                data_thr['y']%1000,    
                                data_thr['z']%1000]).T

        D1_x   = copy.deepcopy(gal_pos_thr[:, 0])
        D1_y   = copy.deepcopy(gal_pos_thr[:, 1])
        D1_z   = copy.deepcopy(gal_pos_thr[:, 2])

        Ng8_thr = N_g_8_theory( D1_x, D1_y, D1_z )
        ### Ng8 bottom 20% #########################################################################################################
        D2_x = select_bottom20_in_random(D1_x, Ng8_thr)
        D2_y = select_bottom20_in_random(D1_y, Ng8_thr)
        D2_z = select_bottom20_in_random(D1_z, Ng8_thr)
        weights = np.ones_like(D2_x)

        CCMD_box_xi_r_bot20 = Corrfunc.theory.xi(boxsize = 1000, nthreads = 50, binfile = r_bin, 
                                                X = D2_x, Y = D2_y, Z = D2_z,
                                                weights=weights, weight_type='pair_product', output_ravg=True )['xi']
        ### Ng8 top 80% ############################################################################################################
        D2_x = select_top80_in_random(D1_x, Ng8_thr)
        D2_y = select_top80_in_random(D1_y, Ng8_thr)
        D2_z = select_top80_in_random(D1_z, Ng8_thr)
        weights = np.ones_like(D2_x)

        CCMD_box_xi_r_top80 = Corrfunc.theory.xi(boxsize = 1000, nthreads = 50, binfile = r_bin, 
                                                X = D2_x, Y = D2_y, Z = D2_z,
                                                weights=weights, weight_type='pair_product', output_ravg=True )['xi']
        #############################################################################################################################
        
        output_dict1['xi_r_bot20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = CCMD_box_xi_r_bot20
        output_dict2['xi_r_top80_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = CCMD_box_xi_r_top80

        np.savez('./CCMD_box_Mr_C_xi_bottom_20.npz', **output_dict1)
        np.savez('./CCMD_box_Mr_C_xi_top_80.npz', **output_dict2)


In [ ]:
CCMD_box_xi_r       = np.load('./CCMD_box_Mr_C_xi.npz')
CCMD_box_xi_r_bot20 = np.load('./CCMD_box_Mr_C_xi_bottom_20.npz')
CCMD_box_xi_r_top80 = np.load('./CCMD_box_Mr_C_xi_top_80.npz')

xi_mm_r = np.array([0.3257651 , 0.08987048, 0.03611578, 0.01660525, 0.00844831])
r_bin = np.array([5, 15, 25, 35, 45, 55])
r_bin_x = (r_bin[1:] + r_bin[:-1])/2

In [ ]:
fig, ax = plt.subplots(4, 4, figsize = (20, 12), sharex = True, sharey = True)
plt.subplots_adjust(hspace = 0, wspace = 0)

pimax_texts    = [r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 60$', r'$\pi_{\rm max} = 40$']
x_add = [1.5, -0.65, -0.65, 0.1]

for j in range(4):
    for i in range(4):
    ### CCMD 32Quads #########################################################################################
        xi_gg_r    = CCMD_box_xi_r['xi_r_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
        bg_square  = xi_gg_r / xi_mm_r
        bias_g_all = np.sqrt(bg_square)

        bias_g_bot = np.sqrt(CCMD_box_xi_r_bot20['xi_r_bot20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] / xi_mm_r)
        bias_g_top = np.sqrt(CCMD_box_xi_r_top80['xi_r_top80_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] / xi_mm_r)
        
        # ax[j,i].plot(r_bin_x, xi_gg_r, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
        ax[j,i].plot(r_bin_x, bias_g_all, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD all')
        ax[j,i].plot(r_bin_x, bias_g_bot, color = 'orangered'  , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD bottom 20%')
        ax[j,i].plot(r_bin_x, bias_g_top, color = 'royalblue'  , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD top 80%')
#############################################################################################################   
        # ax[j,i].set_yscale('log')
        ax[j,i].tick_params(axis = 'x', direction = 'in', which = u'both')
        ax[j,i].tick_params(axis = 'y', direction = 'in', which = u'both')
        ax[j,i].set_ylim(0.0, 2.2)
        ax[j,i].text(10, 0.3, r'$b_g^{all} = $' + '{:.2f}'.format(np.nanmean(bias_g_all)), fontsize=15,)
        ax[j,i].text(24, 0.3, r'$b_g^{bot} = $' + '{:.2f}'.format(np.nanmean(bias_g_bot)), fontsize=15,)
        ax[j,i].text(38, 0.3, r'$b_g^{top} = $' + '{:.2f}'.format(np.nanmean(bias_g_top)), fontsize=15,)

    
for j in range(4):
    # ax[j,0].set_ylabel(r'$\xi_{gg}(r)$',fontsize = 16,)
    ax[j,0].set_ylabel(r'$b = \sqrt{\xi_{gg}(r)/\xi_{mm}(r)}$',fontsize = 16,)
    ax[j,3].text(55, 1.25 , Mr_thr_text[j], rotation=90, va='center', ha='left', fontsize=16,)
    # ax[0,0].text(11.2, 1.3e3, 'bluest', fontsize = 16, color = 'royalblue')
    # ax[0,3].text(9.8, 1.3e3, 'reddest', fontsize = 16, color = 'red')

for i in range(4):
    ax[3,i].set_xlabel(r'$r\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)

legend = ax[3,0].legend(fontsize = 14, loc = 'upper left',frameon = False) 
legend.get_texts()[0].set_color('forestgreen')
legend.get_texts()[1].set_color('orangered')
legend.get_texts()[2].set_color('royalblue')

# plt.suptitle(r'$N_g^8$ bottom $20\%$', y =0.95)
# plt.savefig('../../../Figures/fig_data_Mr_C_wp_Ng8_bottom20.pdf', bbox_inches='tight')

In [ ]:
r_bin   = np.array([4, 6, 8, 10, 12])
r_bin_x = (r_bin[1:] + r_bin[:-1])/2
r_list = ['r_4_6', 'r_6_8', 'r_8_10', 'r_10_12']
r_text = [r'$r: [4, 6]$',r'$r: [6, 8]$',r'$r: [8, 10]$',r'$r: [10, 12]$',]

data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data

xi_mm_r = np.array([0.97387602, 0.57831325, 0.38657351, 0.27625475]) # r: array([ 5.,  7.,  9., 11.])

Mr_bins    = np.linspace(-22, -18, 21)
C_bins     = np.linspace(0, 1.2, 25)

output_dict1 = collections.defaultdict(list)

for k in range(len(r_bin)-1):
    r_bin_temp = np.array([r_bin[k], r_bin[k+1]])
    CCMD_box_xi_r  = np.zeros(( len(Mr_bins) - 1, len(C_bins) - 1 ))

    for j in range(len(Mr_bins)-1): # Mr[-22,-18] 
        for i in range(len(C_bins)-1): # C[0, 1.2]
            mask = np.logical_and(  np.logical_and(Mr_bins[j] < data['Mr'] , data['Mr']  < Mr_bins[j+1]),
                                    np.logical_and(C_bins[i]  < data['gmr'], data['gmr'] < C_bins[i+1] ))
            data_thr  = data[mask]

            gal_pos_thr = np.array([data_thr['x']%1000,
                                    data_thr['y']%1000,    
                                    data_thr['z']%1000]).T

            D1_x   = copy.deepcopy(gal_pos_thr[:, 0])
            D1_y   = copy.deepcopy(gal_pos_thr[:, 1])
            D1_z   = copy.deepcopy(gal_pos_thr[:, 2])

            weights = np.ones_like(data_thr['x'])
            
            CCMD_box_xi_r[j,i] = Corrfunc.theory.xi(boxsize = 1000, nthreads = 50, binfile = r_bin_temp, 
                                                X = gal_pos_thr[:, 0], Y = gal_pos_thr[:, 1], Z = gal_pos_thr[:, 2],
                                                weights=weights, weight_type='pair_product', output_ravg=True )['xi']
            
    CCMD_box_bias      = CCMD_box_xi_r/xi_mm_r[k]
            
    output_dict1['bias_' + r_list[k]] = CCMD_box_bias
    np.savez('./CCMD_box_bias_Mr_C_linear_bias.npz', **output_dict1)
            


In [ ]:
# fig, ax = plt.subplots(4, 1, figsize = (20, 3), sharex = True, sharey = True)
# plt.subplots_adjust(hspace = 0, wspace = 0)
# Mr_bins    = np.linspace(-22, -18, 21)
# C_bins     = np.linspace(0, 1.2, 25)


# for k in range(4):
#     r_bin = np.array([5, 7, 9, 11])

#     for j in range(len(Mr_bins)-1): # Mr[-22,-18] 
#         for i in range(len(C_bins)-1): # C[0, 1.2]
# ### CCMD 32Quads #########################################################################################
#             xi_gg_r    = CCMD_box_xi_r['xi_r_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]]
#             bg_square  = xi_gg_r / xi_mm_r
#             bias_g_all = np.sqrt(bg_square)

#             bias_g_bot = np.sqrt(CCMD_box_xi_r_bot20['xi_r_bot20_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] / xi_mm_r)
#             bias_g_top = np.sqrt(CCMD_box_xi_r_top80['xi_r_top80_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] / xi_mm_r)
            
#             # ax[j,i].plot(r_bin_x, xi_gg_r, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD')
#             ax[j].plot(r_bin_x, bias_g_all, color = 'forestgreen', alpha = 0.9, lw = 3, ls = '-', label = 'CCMD all')
#             ax[j].plot(r_bin_x, bias_g_bot, color = 'orangered'  , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD bottom 20%')
#             ax[j].plot(r_bin_x, bias_g_top, color = 'royalblue'  , alpha = 0.9, lw = 3, ls = '-', label = 'CCMD top 80%')
# #############################################################################################################   
#             # ax[j,i].set_yscale('log')
#             ax[j].tick_params(axis = 'x', direction = 'in', which = u'both')
#             ax[j].tick_params(axis = 'y', direction = 'in', which = u'both')
#             # ax[j].set_ylim(0.0, 2.2)
#             # ax[j].text(10, 0.3, r'$b_g^{all} = $' + '{:.2f}'.format(np.nanmean(bias_g_all)), fontsize=15,)
#             ax[j].set_xlabel(r'$r\ [h^{-1}{\rm Mpc}]$',fontsize = 16,)
#             ax[0].set_ylabel(r'$b = \sqrt{\xi_{gg}(r)/\xi_{mm}(r)}$',fontsize = 16,)

#     ax[k].title(r_text[k])

# legend = ax[0,0].legend(fontsize = 14, loc = 'upper left',frameon = False) 
# legend.get_texts()[0].set_color('forestgreen')

In [ ]:
all_data = np.load('./CCMD_box_bias_Mr_C_linear_bias.npz')
all_data.files

In [ ]:
def resample_1percent(data):
    np.random.seed(42)
    
    n_total  = len(data)
    n_select = int(np.round(n_total * 0.02))
    # n_select = int(np.round(n_total * 0.0002))

    # 方法1：创建全False数组，然后随机设置True
    mask = np.zeros(n_total, dtype=bool)
    selected_indices = np.random.choice(n_total, size=n_select, replace=False)
    mask[selected_indices] = True

    result = data[mask]

    return result


In [ ]:
### Linear bias
all_data = np.load('./CCMD_box_bias_Mr_C_linear_bias.npz')

data = all_data['bias_r_4_6']  # 示例数据，范围[0, 1.2]
data_processed = data.copy()
data_processed[data_processed <= 0] = np.nan
data_processed = np.log10(data_processed)
# 创建横纵坐标轴
x = np.linspace(-22, -18, 20)  # 横轴 [-18, -22] 分20个间隔
y = np.linspace(0, 1.2, 24)    # 纵轴 [0, 1.2] 分24个间隔

# 创建网格
X, Y = np.meshgrid(x, y)

# 绘制等高线图
plt.figure(figsize=(10, 8))
contour = plt.contourf(X, Y, data_processed.T, levels=50, cmap='rainbow')  # 转置数据以匹配网格

# 添加colorbar
cbar = plt.colorbar(contour)
cbar.set_label('Log(bias)', fontsize=12)

### Mr-gmr
gal_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
Mr  = resample_1percent(gal_data['Mr']) # host halo mass, log(Msun/h)
gmr = resample_1percent(gal_data['gmr'])
kde   = gaussian_kde([gmr, Mr])
Z     = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)
plt.contour(X, Y, Z, colors='black')


# 设置坐标轴标签和标题
plt.xlabel('Mr', fontsize=12)
plt.ylabel('g-r', fontsize=12)
plt.title(r'$r \in [4, 6]$', fontsize=14)
plt.gca().invert_xaxis()

# 显示网格
plt.grid(True, alpha=0.3)

# 显示图形
plt.tight_layout()
plt.show()

In [ ]:
all_data.files

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

### Linear bias
all_data = np.load('./CCMD_box_bias_Mr_C_linear_bias.npz')

data_keys = ['bias_r_4_6', 'bias_r_6_8', 'bias_r_8_10', 'bias_r_10_12']
titles = [r'$r \in [4, 6]$', r'$r \in [6, 8]$', r'$r \in [8, 10]$', r'$r \in [10, 12]$']

# 创建1行4列的子图，设置间距为0
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharex=True, sharey=True)
plt.subplots_adjust(wspace=0.)  # 设置水平间距为0

# 坐标轴
x = np.linspace(-22, -18, 20)
y = np.linspace(0, 1.2, 24)
X, Y = np.meshgrid(x, y)

# 预处理所有数据并计算全局范围
all_processed_data = []
for key in data_keys:
    data = all_data[key]
    data_processed = data.copy()
    data_processed[data_processed <= 0] = np.nan
    data_processed = np.log10(data_processed)
    all_processed_data.append(data_processed)

vmin = min([np.nanmin(data) for data in all_processed_data])
vmax = max([np.nanmax(data) for data in all_processed_data])

# 绘制子图
im = None
for i, (ax, key, title) in enumerate(zip(axes, data_keys, titles)):
    data_processed = all_processed_data[i]
    

    im = ax.contourf(X, Y, data_processed.T, levels=50, cmap='rainbow', vmin=vmin, vmax=vmax)
    ax.set_title(title, fontsize=13)
    ax.grid(True, alpha=0.3)
    
    if i == 0:
        # 最左侧子图显示完整的x轴标签（包含-18）
        ax.set_xticks([-22, -21, -20, -19, -18])
        ax.set_xticklabels(['-22','-21', '-20', '-19', '-18'])
        ax.set_ylabel(r'$g-r$', fontsize=16)
    else:
        # 其他子图不显示-18
        ax.set_xticks([-22, -21, -20, -19])
        ax.set_xticklabels(['-22','-21', '-20', '-19'])
        ax.set_yticklabels([])
    
    # 所有子图都显示x轴标签
    ax.set_xlabel(r'$M_r$', fontsize=16)
        

# 调整布局，为colorbar留出足够空间
plt.subplots_adjust(right=0.85)  # 为colorbar留出15%的空间
plt.gca().invert_xaxis()

# 添加共用colorbar，确保不重叠
cbar_ax = fig.add_axes([0.87, 0.15, 0.02, 0.7])  # 调整位置，确保不重叠
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Log(bias)', fontsize=11)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# 加载数据
all_data = np.load('./CCMD_box_bias_Mr_C_linear_bias.npz')

data_keys = ['bias_r_4_6', 'bias_r_6_8', 'bias_r_8_10', 'bias_r_10_12']
titles = [r'$r \in [4, 6]$', r'$r \in [6, 8]$', r'$r \in [8, 10]$', r'$r \in [10, 12]$']

# 创建1行4列的子图
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharex=True, sharey=True)
plt.subplots_adjust(wspace=0.)  # 设置水平间距为0

# 坐标轴网格
x = np.linspace(-22, -18, 20)
y = np.linspace(0, 1.2, 24)
X, Y = np.meshgrid(x, y)

# 定义颜色范围
vmin, vmax = 0.3, 5.0

# 用于存储所有contourf对象，以便创建colorbar
contours = []

# 绘制子图
for i, (ax, key, title) in enumerate(zip(axes, data_keys, titles)):
    # 获取原始数据
    data = all_data[key]
    
    # 创建数据副本，将<=0的值设为NaN
    data_processed = data.copy()
    data_processed[data_processed <= 0] = np.nan
    
    # 创建掩码，但保持二维结构
    mask = (data_processed > 0.3) & (data_processed < 5.0)
    
    # 应用掩码：将超出范围的值设为NaN
    data_to_plot = data_processed.copy()
    data_to_plot[~mask] = np.nan
    
    # 绘制contourf图
    contour = ax.contourf(X, Y, data_to_plot.T, levels=50, cmap='rainbow', vmin=vmin, vmax=vmax)
    contours.append(contour)
    
    ax.set_title(title, fontsize=13)
    ax.grid(True, alpha=0.3)
    
    if i == 0:
        # 最左侧子图显示完整的x轴标签
        ax.set_xticks([-22, -21, -20, -19, -18])
        ax.set_xticklabels(['-22', '-21', '-20', '-19', '-18'])
        ax.set_ylabel(r'$g-r$', fontsize=16)
    else:
        # 其他子图不显示-18
        ax.set_xticks([-22, -21, -20, -19])
        ax.set_xticklabels(['-22', '-21', '-20', '-19'])
        ax.set_yticklabels([])
    
    # 所有子图都显示x轴标签
    ax.set_xlabel(r'$M_r$', fontsize=16)

# 反转x轴
plt.gca().invert_xaxis()

# 调整布局，为colorbar留出空间
plt.subplots_adjust(right=0.85)

# 添加共用colorbar（使用最后一个contourf对象）
cbar_ax = fig.add_axes([0.87, 0.15, 0.02, 0.7])
cbar = fig.colorbar(contours[-1], cax=cbar_ax)
cbar.set_label('Linear bias', fontsize=11)

plt.show()

In [ ]:
gal_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
Mr  = resample_1percent(gal_data['Mr']) # host halo mass, log(Msun/h)
gmr = resample_1percent(gal_data['gmr'])

Mr  = resample_1percent(Mr) # host halo mass, log(Msun/h)
gmr = resample_1percent(gmr)

# 计算密度
# kde   = gaussian_kde([Mr[mask], gmr[mask]])
kde   = gaussian_kde([Mr, gmr])
xgrid = np.linspace(-22, -18, 20)  # 调整X范围
ygrid = np.linspace(0.0, 1.2, 20)  # 调整Y范围

X, Y  = np.meshgrid(xgrid, ygrid)
Z     = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

# 创建图形
plt.figure(figsize=(10, 8))#,dpi=100

contours = plt.contour(X, Y, Z, levels=9, colors='black')

# 设置坐标轴
plt.xlabel(r'$M_r$', fontsize=34, labelpad = 10)
plt.ylabel(r'$g-r$', fontsize=34, y = 0.01)
plt.gca().invert_xaxis()

plt.xticks([-22, -21, -20, -19, -18.02], 
           ['-22', '-21', '-20', '-19', '-18'])
plt.yticks([0.0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2], 
           ['0', '0.2', '0.4', '0.6', '0.8', '1.0', '1.2'])

plt.tick_params(
    axis='both',        # 同时设置x和y轴
    which='major',      # 设置主刻度
    direction='in',     # 刻度方向：'in'内侧, 'out'外侧, 'inout'两侧
    length=9,           # 刻度线长度
    width=3,            # 刻度线粗细
    labelsize=24        # 刻度标签字体大小
)

plt.xlim(-18, -22)
plt.ylim(0.0, 1.2)

plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from astropy.io import fits
from scipy.stats import gaussian_kde

# 加载bias数据
all_data = np.load('./CCMD_box_bias_Mr_C_linear_bias.npz')

data_keys = ['bias_r_4_6', 'bias_r_6_8', 'bias_r_8_10', 'bias_r_10_12']
titles = [r'$r \in [4, 6]$', r'$r \in [6, 8]$', r'$r \in [8, 10]$', r'$r \in [10, 12]$']

# 加载星系数据并计算KDE
gal_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
   
# 获取并重采样数据
Mr = resample_1percent(gal_data['Mr'])
gmr = resample_1percent(gal_data['gmr'])

# 计算KDE密度
kde = gaussian_kde([Mr, gmr])
x_kde = np.linspace(-22, -18, 20)
y_kde = np.linspace(0.0, 1.2, 20)
X_kde, Y_kde = np.meshgrid(x_kde, y_kde)
Z_kde = kde(np.vstack([X_kde.ravel(), Y_kde.ravel()])).reshape(X_kde.shape)

# 创建1行4列的子图
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), )
plt.subplots_adjust(wspace=0.)

# bias图的坐标轴网格
x_bias = np.linspace(-22, -18, 20)
y_bias = np.linspace(0, 1.2, 24)
X_bias, Y_bias = np.meshgrid(x_bias, y_bias)

# 定义颜色范围
vmin, vmax = 0.3, 5.0

# 用于存储所有contourf对象
contours_bias = []

# 绘制子图
for i, (ax, key, title) in enumerate(zip(axes, data_keys, titles)):
    # 获取原始bias数据
    data = all_data[key]
    
    # 创建数据副本，将<=0的值设为NaN
    data_processed = data.copy()
    data_processed[data_processed <= 0] = np.nan
    
    # 创建掩码，但保持二维结构
    mask = (data_processed > 0.3) & (data_processed < 5.0)
    
    # 应用掩码：将超出范围的值设为NaN
    data_to_plot = data_processed.copy()
    data_to_plot[~mask] = np.nan
    
    # 绘制bias contourf图
    contour_bias = ax.contourf(X_bias, Y_bias, data_to_plot.T, 
                               levels=50, cmap='rainbow', vmin=vmin, vmax=vmax)
    contours_bias.append(contour_bias)
    
    # 叠加KDE等值线
    contours_kde = ax.contour(X_kde, Y_kde, Z_kde, 
                              levels=9, colors='black', linewidths=1.5, alpha=0.8)
    
    # 设置标题和网格
    ax.set_title(title, fontsize=13)
    ax.grid(True, alpha=0.3)
    
    # 关键修改：为每个子图单独反转x轴
    ax.invert_xaxis()
    
    # 设置坐标轴标签和刻度
    if i == 0:
        # 最左侧子图显示完整的x轴标签
        ax.set_xticks([-22, -21, -20, -19, -18])
        ax.set_xticklabels(['-22', '-21', '-20', '-19', '-18'])
        ax.set_ylabel(r'$g-r$', fontsize=16)
        ax.set_yticks([0, 0.2, 0.4, 0.6, 0.8, 1.0, 1.2])
        ax.set_yticklabels(['0.0', '0.2', '0.4', '0.6', '0.8', '1.0', '1.2'])
    else:
        # 其他子图不显示-18
        ax.set_xticks([-22, -21, -20, -19])
        ax.set_xticklabels(['-22', '-21', '-20', '-19'])
        ax.set_yticklabels([])
    
    # 所有子图都显示x轴标签
    ax.set_xlabel(r'$M_r$', fontsize=16)
    
    # 设置坐标轴范围（反转后需要正确设置）
    ax.set_xlim(-18, -22)  # 注意：这里范围是(-18, -22)因为已经反转了
    ax.set_ylim(0, 1.2)

# 调整布局，为colorbar留出空间
plt.subplots_adjust(right=0.85)

# 添加共用colorbar
cbar_ax = fig.add_axes([0.87, 0.15, 0.02, 0.7])
cbar = fig.colorbar(contours_bias[-1], cax=cbar_ax)
cbar.set_label('Linear bias', fontsize=11)

plt.show()

In [ ]:
np.nanmax(all_processed_data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde
from astropy.io import fits

### Linear bias
all_data = np.load('./CCMD_box_bias_Mr_C_linear_bias.npz')

data_keys = ['bias_r_4_6', 'bias_r_6_8', 'bias_r_8_10', 'bias_r_10_12']
titles = [r'$r \in [4, 6]$', r'$r \in [6, 8]$', r'$r \in [8, 10]$', r'$r \in [10, 12]$']

# 创建1行4列的子图，设置间距为0
fig, axes = plt.subplots(1, 4, figsize=(16, 4.5), sharex=True, sharey=True)
plt.subplots_adjust(wspace=0)

# 坐标轴
x = np.linspace(-22, -18, 20)
y = np.linspace(0, 1.2, 24)
X, Y = np.meshgrid(x, y)

# 预处理所有数据并计算全局范围
all_processed_data = []
for key in data_keys:
    data = all_data[key]
    data_processed = data.copy()
    data_processed[data_processed <= 0] = np.nan
    data_processed = np.log10(data_processed)
    all_processed_data.append(data_processed)

vmin = min([np.nanmin(data) for data in all_processed_data])
vmax = max([np.nanmax(data) for data in all_processed_data])

# 加载并处理第二份数据用于黑线轮廓
gal_data = fits.open("/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits")[1].data
Mr_data = resample_1percent(gal_data['Mr'])  # host halo mass, log(Msun/h)
gmr_data = resample_1percent(gal_data['gmr'])
kde = gaussian_kde([gmr_data, Mr_data])
Z = kde(np.vstack([X.ravel(), Y.ravel()])).reshape(X.shape)

# 绘制子图
im = None
for i, (ax, key, title) in enumerate(zip(axes, data_keys, titles)):
    data_processed = all_processed_data[i]
    
    # 绘制主要的等高线填充图
    im = ax.contourf(X, Y, data_processed.T, levels=50, cmap='rainbow', vmin=vmin, vmax=vmax)
    
    # 叠加第二份数据的黑线轮廓
    ax.contour(X, Y, Z, colors='black', linewidths=1.5, alpha=0.8)
    
    ax.set_title(title, fontsize=13)
    ax.grid(True, alpha=0.3)
    ax.invert_xaxis()
    
    # 设置x轴范围确保包含-18
    ax.set_xlim(-22.2, -17.8)
    
    # 设置x轴刻度
    if i == 0:
        ax.set_ylabel(r'$g-r$', fontsize=16)
        ax.set_xticks([-22, -21, -20, -19, -18])
        ax.set_xticklabels(['-22','-21', '-20', '-19', '-18'])
    else:
        ax.set_yticklabels([])
        ax.set_xticks([-22, -21, -20, -19])
        ax.set_xticklabels(['-22','-21', '-20', '-19'])
    
    ax.set_xlabel('Mr', fontsize=11)

# 调整布局，为colorbar留出空间
plt.subplots_adjust(right=0.85)

# 添加共用colorbar
cbar_ax = fig.add_axes([0.87, 0.15, 0.02, 0.7])
cbar = fig.colorbar(im, cax=cbar_ax)
cbar.set_label('Log(bias)', fontsize=11)

plt.show()